# 05 — Ideal-state QAOA on the frozen path-QUBO benchmarks

The preceding notebooks separated the physical OEC model, the direct QUBO encoding, the candidate-path reduction and the structural resource analysis. The present notebook introduces QAOA only after these layers have been validated independently. In particular, every benchmark considered here has an exact classical optimum and a zero decomposition gap at the retained candidate-set size, so the variational error can be interpreted without conflating it with an approximation introduced by the classical reduction.

The experiment uses an ideal statevector implementation acting directly on the explicit path-QUBO Hamiltonian. No shot noise or hardware noise is introduced in this stage. The objective is therefore to characterize the optimization behavior of finite-depth QAOA under controlled conditions before the effects of finite sampling are studied separately.

In [ ]:
from __future__ import annotations

import pandas as pd

from oec_qaoa.benchmarks import (
    FROZEN_QAOA_BENCHMARKS,
    build_frozen_benchmark,
    physical_feasibility_mask,
)
from oec_qaoa.qaoa import optimize_qaoa

DEPTHS = [1, 2, 3, 4]
RESTART_SEEDS = [101, 202, 303]
MAXITER = 100

## Frozen benchmark set

The benchmark set was selected in a purely classical stage from exact-optimum-known synthetic OEC instances. Each selected path reduction satisfies (Delta_K=0), which ensures that the reduced constrained model retains the optimum of the corresponding full edge formulation. The selected QUBOs span approximately (10) to (16) logical variables, providing a range that remains tractable for repeated ideal statevector simulations while being large enough to expose non-trivial interaction structure.

In [ ]:
benchmark_rows = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    benchmark_rows.append(
        {
            "benchmark_id": benchmark.benchmark_id,
            "instance": instance.name,
            "K": benchmark.k,
            "logical_variables": len(model.variable_names),
            "semantic_variables": model.semantic_count,
            "quadratic_couplings": len(model.quadratic),
        }
    )

pd.DataFrame(benchmark_rows)

## Variational protocol

For a path-QUBO Hamiltonian (H_C), the depth-(p) QAOA state is

[
|psi_p(oldsymbol{gamma},oldsymbol{eta})angle
=
prod_{ell=1}^{p}
e^{-ieta_ell H_M}
e^{-igamma_ell H_C}
|+angle^{otimes n},
]

with the standard transverse-field mixer (H_M=sum_i X_i). The cost unitary is applied exactly in the computational basis from the QUBO energies, while the mixer factorizes into single-qubit (X) rotations. This gives an exact ideal-state simulation for the problem sizes used here.

Each depth is optimized from three deterministic random initializations using COBYLA. The principal quantities reported are the expected QUBO energy, the probability of the exact QUBO optimum and the probability that the semantic path decisions correspond to a physically feasible OEC allocation. Slack variables are marginalized in the latter quantity because they are encoding variables rather than physical decisions.

In [ ]:
records = []

for benchmark in FROZEN_QAOA_BENCHMARKS:
    instance, retained, model = build_frozen_benchmark(benchmark)
    feasible_mask = physical_feasibility_mask(instance, retained, model)

    for depth in DEPTHS:
        for restart, seed in enumerate(RESTART_SEEDS, start=1):
            result = optimize_qaoa(
                model,
                depth=depth,
                seed=seed,
                feasible_mask=feasible_mask,
                optimizer="COBYLA",
                maxiter=MAXITER,
            )

            records.append(
                {
                    "benchmark_id": benchmark.benchmark_id,
                    "depth": depth,
                    "restart": restart,
                    "seed": seed,
                    "expected_energy": result.expected_energy,
                    "optimal_energy": result.optimal_energy,
                    "expected_gap": (
                        result.expected_energy - result.optimal_energy
                    ),
                    "optimal_probability": result.optimal_probability,
                    "feasible_probability": result.feasible_probability,
                    "most_likely_energy": result.most_likely_energy,
                    "n_evaluations": result.n_evaluations,
                    "optimizer_success": result.success,
                }
            )

runs = pd.DataFrame(records)
runs.head()

## Aggregated results

The restart-level data are retained because the optimizer can converge to different parameter regions even when the quantum state evolution itself is deterministic. The summary below therefore reports both the mean behavior across restarts and the best value obtained at each depth. These should not be interpreted as hardware success probabilities: they are exact statevector probabilities for the ideal QAOA state after classical parameter optimization.

In [ ]:
summary = (
    runs.groupby(["benchmark_id", "depth"])
    .agg(
        expected_energy_mean=("expected_energy", "mean"),
        expected_energy_best=("expected_energy", "min"),
        expected_gap_mean=("expected_gap", "mean"),
        optimal_probability_mean=("optimal_probability", "mean"),
        optimal_probability_best=("optimal_probability", "max"),
        feasible_probability_mean=("feasible_probability", "mean"),
        feasible_probability_best=("feasible_probability", "max"),
        n_evaluations_mean=("n_evaluations", "mean"),
    )
    .reset_index()
)

summary

## Interpretation protocol

Three effects are kept separate when these results are discussed. First, the exact benchmark optimum is fixed by the classical reference model. Second, the candidate-path reduction contributes no objective gap for the frozen benchmarks. Third, the quantities observed here measure only the finite-depth variational optimization of the resulting QUBO.

An increase of (p) is therefore useful only if it improves the probability mass on optimal or feasible states at a computational cost that remains compatible with the classical optimization budget. Conversely, a low (P_{mathrm{opt}}) at shallow depth cannot be attributed to the path reduction on this benchmark set, because the exact optimum remains present in every selected reduced model.

The next stage will repeat the same benchmark protocol with finite sampling. That experiment will distinguish the ideal variational limitation observed here from the additional uncertainty introduced when expectation values and output distributions are estimated from a finite number of circuit shots.